In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
# Read the "Ontgraven" section (left part)
df_ontgraven = pd.read_excel("../Data/Grondstromen Dashboard Uitvoering.xlsx", sheet_name="Kopie matrix 6.0", 
                             skiprows=14, usecols="A:H", nrows = 273)

# Read the "Aanvullen" section (right part, starting from column I)
# We'll need to transpose this data
df_aanvullen_raw = pd.read_excel("../Data/Grondstromen Dashboard Uitvoering.xlsx", sheet_name="Kopie matrix 6.0", 
                                usecols="H:FD", nrows= 14, header= None)

# Transpose the aanvullen data
df_aanvullen = df_aanvullen_raw.transpose()

headers = df_aanvullen.iloc[0].dropna()

df_aanvullen = df_aanvullen.iloc[8:].drop(columns = [1,2,3,4,5,6])

df_aanvullen.columns = headers

df_aanvullen = df_aanvullen.rename(columns = {'Aanvullen' : 'Object'})

# df_ontgraven = df_ontgraven[df_ontgraven['Locatietype'] != 'Tijdelijk']
# df_aanvullen = df_aanvullen[df_aanvullen['Locatietype'] != 'Tijdelijk']

df_ontgraven['Unnamed: 0'] = df_ontgraven['Unnamed: 0'].fillna(method='ffill')

df_ontgraven = df_ontgraven.rename(columns = {'Unnamed: 0' : 'Object'})

df_ontgraven = df_ontgraven.iloc[1:267]

C:\Users\luke\AppData\Local\Temp\ipykernel_17792\1377877089.py:24: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ontgraven['Unnamed: 0'] = df_ontgraven['Unnamed: 0'].fillna(method='ffill')


In [3]:
df_aanvullen = df_aanvullen[df_aanvullen['Erosieklasse'] != 'zandbed']

In [4]:
df_ontgraven['Object'] = df_ontgraven['Object'].str.upper()
df_aanvullen['Object'] = df_aanvullen['Object'].str.upper()

In [5]:
print(df_ontgraven.head(5))
print(df_ontgraven.tail(5))
print(df_aanvullen.head(5))
print(df_aanvullen.tail(5))

print(f'\nAmount of supply instances: {len(df_ontgraven)}')
print(f'Amount of demand instances: {len(df_aanvullen)}')

  Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
1    NaN   COB-0055     Locatie    WB       klei          NaN          NaN   
2   DS01  OBJ-01.01     Locatie    WB  teelaarde          NaN          NaN   
3   DS01  OBJ-01.01     Locatie    LB  teelaarde          NaN          NaN   
4   DS01  OBJ-01.01     Locatie    WB      grond          NaN          NaN   
5   DS01  OBJ-01.01     Locatie    LB      grond          NaN          NaN   

    Volume  
1 -54000.0  
2  -5953.0  
3  -2167.0  
4  -9282.0  
5  -3994.0  
    Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
262    DUD  OBJ-03.06     Locatie    WB       klei            3           AW   
263    DUD  OBJ-03.06     Locatie    WB       klei            1           AW   
264    DUD  OBJ-03.06     Locatie    WB       klei            1   B (LB=Ind)   
265    DUD  OBJ-03.06     Locatie    WB       klei            3   B (LB=Ind)   
266    DUD  OBJ-03.06     Locatie    WB       klei   

In [6]:
# COB klei is klei NG 

df_ontgraven['Erosieklasse'] = df_ontgraven.apply(lambda x: '3' if x['Locatie'] == 'COB-0055' else x['Erosieklasse'], axis=1)

In [7]:
# transport loss + 10% loss for teelaarde
df_ontgraven['Volume'] = df_ontgraven.apply(
    lambda x: x['Volume'] * 0.88 if x['Grondsoort'] == 'teelaarde' 
              else x['Volume'] if x['Erosieklasse'] == 'Keramische klei' 
              else x['Volume'] * 0.98, 
    axis=1
)

In [8]:
#Add column to work with compatibility tables correctly

df_ontgraven['Grondsoort'] = df_ontgraven['Grondsoort'].astype(str)
df_ontgraven['Erosieklasse'] = df_ontgraven['Erosieklasse'].astype(str)

df_ontgraven['Civieltechnisch'] = df_ontgraven['Grondsoort'] + ' ' + df_ontgraven['Erosieklasse']

df_ontgraven['Civieltechnisch'] = df_ontgraven['Civieltechnisch'].str.replace('nan', '').str.strip()

df_ontgraven['Civieltechnisch'] = df_ontgraven['Civieltechnisch'].replace('klei Keramische klei', 'Keramische klei')

print(df_ontgraven['Civieltechnisch'].unique())

['klei 3' 'teelaarde' 'grond' 'teelaarde_kleiinkassing' 'zand NG'
 '<5 klei NG' '<5 klei 3' 'klei NG' '<5 klei 1' '<5 klei 2' 'klei 2'
 'klei 1' 'zand werkweg' 'klei' 'klei 1-aaltje' 'klei 2-aaltje'
 'klei 3-aaltje' 'Keramische klei' 'grind']


In [9]:
df_aanvullen['Grondsoort'] = df_aanvullen['Grondsoort'].astype(str)
df_aanvullen['Erosieklasse'] = df_aanvullen['Erosieklasse'].astype(str)

df_aanvullen['Civieltechnisch'] = df_aanvullen['Grondsoort'] + ' ' + df_aanvullen['Erosieklasse']

df_aanvullen['Civieltechnisch'] = df_aanvullen['Civieltechnisch'].str.replace('nan', '').str.strip()

df_aanvullen['Civieltechnisch'] = df_aanvullen['Civieltechnisch'].replace({
    'zand': 'zand NG',
    'klei Roofgrond': 'Roofgrond',
    'klei_kleiinkassing 1': 'klei_kleiinkassing 3',
    'kernmateriaal 3': 'klei 3',
    'klei Roofgrond flex' : 'Roofgrond flex',
})

print(df_aanvullen['Civieltechnisch'].unique())

['teelaarde' 'klei 3' 'kernmateriaal' 'klei 2' 'teelaarde_kleiinkassing'
 'klei_kleiinkassing 3' 'zand NG' 'klei 1' 'grond' 'klei NG' 'Roofgrond'
 'Roofgrond flex']


In [10]:
print(df_ontgraven['Object'].unique())

df_ontgraven['Objecttype'] = df_ontgraven['Object'].apply(lambda x: 'Dijk' if str(x).startswith('DS') else 'Rivier')

df_ontgraven.loc[1, 'Objecttype'] = 'Dijk'

print(df_ontgraven[['Object', 'Objecttype']].head(5))
print(df_ontgraven[['Object', 'Objecttype']].tail(5))

[nan 'DS01' 'DS02A' 'DS02B' 'DS03' 'DS04' 'DS05A' 'DS05B' 'DS06' 'DS07A'
 'DS07B' 'DS08' 'DS09A' 'DS09B' 'DS10' 'DS11' 'DAP' 'LDW' 'LOS' 'MBW'
 'MBO' 'OZM' 'DUC' 'MDW' 'DUD']
  Object Objecttype
1    NaN       Dijk
2   DS01       Dijk
3   DS01       Dijk
4   DS01       Dijk
5   DS01       Dijk
    Object Objecttype
262    DUD     Rivier
263    DUD     Rivier
264    DUD     Rivier
265    DUD     Rivier
266    DUD     Rivier


In [11]:
print(df_aanvullen['Object'].unique())

df_aanvullen['Objecttype'] = df_aanvullen['Object'].apply(lambda x: 'Dijk' if str(x).startswith('DS') else 'Rivier')

print(df_aanvullen[['Object', 'Objecttype']].head(5))
print(df_aanvullen[['Object', 'Objecttype']].tail(5))

['DS01' 'DS02A' 'DS02B' 'DS03' 'DS04' 'DS05A' 'DS05B' 'DS06' 'DS07A'
 'DS07B' 'DS08' 'DS09A' 'DS09B' 'DS10' 'DS11' 'DAP' 'LDW' 'LOS' 'MBW'
 'MBO' 'OZM' 'DUC' 'MDW' 'DUD']
7  Object Objecttype
15   DS01       Dijk
16   DS01       Dijk
17   DS01       Dijk
18   DS01       Dijk
19   DS01       Dijk
7   Object Objecttype
155    MDW     Rivier
156    MDW     Rivier
157    DUD     Rivier
158    DUD     Rivier
159    DUD     Rivier


In [12]:
#does ontgraven en aanvullen sum to 0? 

print(df_ontgraven['Volume'].sum())
print(df_aanvullen['Volume'].sum())

print(df_ontgraven['Volume'].sum() + df_aanvullen['Volume'].sum())

-4389900.74
2726514
-1663386.7400000002


In [13]:
print(df_ontgraven['Locatie'].unique())
print(len(df_ontgraven['Locatie'].unique()))

print(df_aanvullen['Locatie'].unique())
print(len(df_aanvullen['Locatie'].unique()))

['COB-0055' 'OBJ-01.01' 'OBJ-01.02' 'OBJ-01.03' 'OBJ-01.04' 'OBJ-01.05'
 'OBJ-01.06' 'OBJ-01.07' 'OBJ-01.08' 'OBJ-01.09' 'OBJ-01.10' 'OBJ-01.11'
 'OBJ-01.12' 'OBJ-01.13' 'OBJ-01.14' 'OBJ-01.15' 'OBJ-02.04' 'OBJ-02.05'
 'OBJ-02.06' 'OBJ-02.07.01' 'OBJ-02.07.02' 'OBJ-02.08' 'OBJ-03.02'
 'OBJ-03.04' 'OBJ-03.06']
25
['OBJ-01.01' 'OBJ-01.02' 'OBJ-01.03' 'OBJ-01.04' 'OBJ-01.05' 'OBJ-01.06'
 'OBJ-01.07' 'OBJ-01.08' 'OBJ-01.09' 'OBJ-01.10' 'OBJ-01.11' 'OBJ-01.12'
 'OBJ-01.13' 'OBJ-01.14' 'OBJ-01.15' 'OBJ-02.04' 'OBJ-02.05' 'OBJ-02.06'
 'OBJ-02.07.01' 'OBJ-02.07.02' 'OBJ-02.08' 'OBJ-03.02' 'OBJ-03.04'
 'OBJ-03.06']
24


# Milieuklasse Check

In [14]:
# Milieuklasse check

# Define the source soil types (VRIJ KOMENDE GROND)
source_soils = [
    ('Dijk', 'LB', 'Industrie (WB = A)'),
    ('Dijk', 'WB', 'A (LB = Ind)'),
    ('Rivier', 'WB', 'NT'),
    ('Rivier', 'WB', 'B (LB = Ind)'),
    ('Rivier', 'WB', 'A (LB = Ind)'),
    ('Rivier', 'WB', 'AW (LB = Ind)'),
    ('Rivier', 'WB', 'AW')
]

# Define the receiving soil types (ONTVANGENDE BODEM/TOEPASSING)
receiving_soils = [
    ('Dijk', 'LB', 'Industrie'),
    ('Dijk', 'WB', 'A (LB = Ind)'),
    ('Rivier', 'WB', 'B (LB = Ind)'),
    ('Rivier', 'WB', 'A (LB = Ind)'),
    ('Rivier', 'WB', 'AW (LB = Ind)'),
    ('Rivier', 'WB', 'AW'),
    ('Algemeen', None, 'Zandbanen'),
    ('Algemeen', None, 'Locale verschraling'),
    ('Algemeen', None, 'Verkoop'),
    ('Algemeen', None, 'Stort')
]

# Create compatibility matrix (True for green/compatible, False for red/incompatible)
# Based on the image pattern
compatibility_matrix = [
    # Dijk, LB, Industrie
    [True, True, True, True, True, True, False, False, False, False],
    # Dijk, WB, A
    [True, True, True, True, True, True, False, False, True, False],
    # Rivier, WB, NT
    [False, False, False, False, False, False, False, False, False, True],
    # Rivier, WB, B
    [True, True, True, True, True, True, True, True, False, False],
    # Rivier, WB, A
    [True, True, True, True, True, True, True, True, False, False],
    # Rivier, WB, AW (LB = Ind)
    [True, True, True, True, True, True, True, True, True, False],
    # Rivier, WB, AW
    [True, True, True, True, True, True, True, True, True, False]
]

# Create a dictionary-based representation for easier lookup
compatibility_dict_env = {}

for i, source in enumerate(source_soils):
    source_key = (source[0], source[1], source[2])
    compatible_targets = []
    
    for j, target in enumerate(receiving_soils):
        if compatibility_matrix[i][j]:
            compatible_targets.append((target[0], target[1], target[2]))
    
    compatibility_dict_env[source_key] = compatible_targets

# Function to check compatibility
def check_compatibility(source, target):
    source_key = (source[0], source[1], source[2])
    if source_key in compatibility_dict_env:
        return target in compatibility_dict_env[source_key]
    return False

check_compatibility(('Dijk', 'LB', 'Industrie'), ('Algemeen', None, 'Zandbanen'))

False

# Erosieklasse check

In [15]:
# Define source soil types (VRIJKOMENDE GROND)
source_soils = [
    # Dijk soils
    ('Dijk', 'teelaarde_kleiinkassing'),
    ('Dijk', 'teelaarde'),
    ('Dijk', 'grond'),
    ('Dijk', 'klei 3'),
    ('Dijk', 'zand NG'),
    # Rivier soils
    ('Rivier', 'klei 1'),
    ('Rivier', 'klei 2'),
    ('Rivier', 'klei 3'),
    ('Rivier', 'klei NG'),
    ('Rivier', '<5 klei 1'),
    ('Rivier', '<5 klei 2'),
    ('Rivier', '<5 klei 3'),
    ('Rivier', '<5 klei NG'),
    ('Rivier', 'klei 1-aaltje'),
    ('Rivier', 'klei 2-aaltje'),
    ('Rivier', 'klei 3-aaltje'),
    ('Rivier', 'Keramische klei'),
    ('Rivier', 'grind'), 
    ('Rivier', 'zand NG'),
    ('Rivier', 'zand werkweg')
]

# Define receiving soil types (TOEPASSING)
receiving_soils = [
    ('Verlies', 'Transportverlies'), #Already doing this upfront in recalculating the Volume column
    ('Verlies', '10% verlies'),
    ('Dijk', 'teelaarde'),
    ('Dijk', 'klei 3'),
    ('Dijk', 'kernmateriaal'),
    ('Dijk', 'klei 2'),
    ('Dijk', 'teelaarde_kleiinkassing'),
    ('Dijk', 'klei_kleiinkassing 3'),
    ('Rivier', 'klei 1'),
    ('Rivier', 'klei 2'),
    ('Rivier', 'klei NG'),
    ('Rivier', 'teelaarde'),
    ('Rivier', 'grond'),
    ('Rivier', 'zand NG'),
    ('Rivier', 'lemig zand'),
    ('Rivier', 'Roofgrond'),
    ('Rivier', 'Roofgrond flex'),
    ('Algemeen', 'Zandbanen'), 
    ('Algemeen', 'Locale verschraling'),
    ('Algemeen', 'Verkoop'),
    ('Algemeen', 'Stort')
]

# Create a matrix representation of the compatibility with priority numbers
# 0 = not compatible (red), 1-6 = priority level (green with number)

compatibility_matrix = [
    # For Dijk - teelaarde_kleininkassing
    [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Dijk - teelaarde
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Dijk - grond
    [1, 0, 0, 1, 3, 0, 0, 2, 0, 0, 0, 0, 4, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Dijk - klei 3
    [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Dijk - zand NG
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0],
    # For Rivier - klei 1
    [1, 0, 0, 5, 6, 1, 0, 2, 3, 4, 8, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Rivier - klei 2
    [1, 0, 0, 4, 6, 1, 0, 2, 0, 3, 7, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Rivier - klei 3
    [1, 0, 4, 2, 3, 0, 0, 1, 0, 0, 6, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Rivier - klei NG
    [1, 0, 2, 0, 1, 0, 3, 0, 0, 0, 5, 4, 1, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Rivier - <5 klei 1
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 4, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Rivier - <5 klei 2
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Rivier - <5 klei 3
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0],
    # For Rivier - <5 klei NG
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Rivier - klei 1-aaltje
    [1, 0, 0, 5, 0, 1, 0, 2, 3, 4, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Rivier - klei 2-aaltje
    [1, 0, 0, 4, 0, 1, 0, 2, 0, 3, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Rivier - klei 3-aaltje
    [1, 0, 4, 2, 3, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0],
    # For Rivier - Keramische klei
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    # For Rivier - grind
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    # For Rivier - zand NG
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0],
    # For Rivier - zand werkweg
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0]
]

# Create a dictionary for easy lookup
compatibility_dict_civ = {}

for i, source in enumerate(source_soils):
    source_key = source
    compatibility_dict_civ[source_key] = {}
    
    for j, target in enumerate(receiving_soils):
        if i < len(compatibility_matrix) and j < len(compatibility_matrix[i]):
            priority = compatibility_matrix[i][j]
            if priority > 0:  # If compatible (has a priority)
                compatibility_dict_civ[source_key][target] = priority

# Convert to pandas DataFrame for easier visualization
index = pd.MultiIndex.from_tuples(source_soils, names=['Location', 'Type'])
columns = pd.MultiIndex.from_tuples(receiving_soils, names=['Destination', 'DestType'])

# Create DataFrame filled with NaN values
df_compatibility = pd.DataFrame(np.nan, index=index, columns=columns)

# Fill in the compatibility values
for i, source in enumerate(source_soils):
    for j, target in enumerate(receiving_soils):
        if i < len(compatibility_matrix) and j < len(compatibility_matrix[i]):
            value = compatibility_matrix[i][j]
            if value > 0:
                df_compatibility.loc[source, target] = value

# Check compatibility and get priority
def check_compatibility(source_location, source_type, dest_location, dest_type):
    source = (source_location, source_type)
    dest = (dest_location, dest_type)
    
    try:
        priority = df_compatibility.loc[source, dest]
        if not pd.isna(priority):
            return True, int(priority)
        else:
            return False, 0
    except KeyError:
        return False, 0
    
# Example usage
source_location = 'Dijk'
source_type = 'teelaarde_kleiinkassing'
dest_location = 'Algemeen'
dest_type = 'Verlies'
is_compatible, priority = check_compatibility(source_location, source_type, dest_location, dest_type)
print(f"Is compatible: {is_compatible}, Priority: {priority}")

Is compatible: False, Priority: 0


In [16]:
print(df_aanvullen.head(5))
print(df_ontgraven.head(5))

7  Object    Locatie Locatietype WB/LB     Grondsoort Erosieklasse  \
15   DS01  OBJ-01.01     Locatie    LB      teelaarde          nan   
16   DS01  OBJ-01.01     Locatie    WB      teelaarde          nan   
17   DS01  OBJ-01.01     Locatie    WB           klei            3   
18   DS01  OBJ-01.01     Locatie    WB  kernmateriaal          nan   
19   DS01  OBJ-01.01     Locatie    LB           klei            3   

7  Milieuklasse Volume Civieltechnisch Objecttype  
15          NaN   2419       teelaarde       Dijk  
16          NaN   6042       teelaarde       Dijk  
17          NaN  13241          klei 3       Dijk  
18          NaN   8886   kernmateriaal       Dijk  
19          NaN   3839          klei 3       Dijk  
  Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
1    NaN   COB-0055     Locatie    WB       klei            3          NaN   
2   DS01  OBJ-01.01     Locatie    WB  teelaarde          nan          NaN   
3   DS01  OBJ-01.01     Locatie  

In [17]:
#find planning in Excel and extract it

df_planning = pd.read_excel("../Data/Grondstromen Dashboard Uitvoering.xlsx", sheet_name="DO leverantie in tijd", 
                             skiprows=4, usecols="D:J", nrows = 25, header= None)

df_planning = df_planning.drop(index = 9, columns = [4, 7])

headers = ['Object', 'Uitvoering_Start', 'Uitvoering_End', 'Grondwerk_Start', 'Grondwerk_End']

df_planning.columns = headers

df_planning['Object'] = df_planning['Object'].str.upper()

print(df_planning)


   Object Uitvoering_Start Uitvoering_End Grondwerk_Start Grondwerk_End
0     LDW       2025-02-01     2026-11-01      2025-04-01    2026-11-01
1     MBO       2025-03-01     2025-10-01      2025-03-01    2025-07-01
2     MDW       2025-03-01     2027-08-01      2025-04-01    2027-08-01
3     DUC       2025-05-01     2030-09-01      2025-05-01    2030-07-01
4     LOS       2025-07-01     2027-11-01      2026-02-01    2027-11-01
5     OZM       2028-06-01     2028-12-01      2026-06-01    2028-11-01
6     MBW       2027-01-01     2028-06-01      2027-01-01    2028-06-01
7     DUD       2028-06-01     2029-01-01      2028-06-01    2028-11-01
8     DAP       2028-06-01     2029-02-01      2028-06-01    2028-12-01
10   DS01       2025-11-01     2027-05-01      2026-02-01    2026-03-01
11  DS02A       2025-07-01     2027-02-01      2025-08-01    2025-11-01
12  DS02B       2025-01-01     2026-11-01      2025-01-01    2025-08-01
13   DS03       2025-09-01     2027-02-01      2025-09-01    202

In [18]:
df_costs = pd.read_excel("../Data/Grondstromen - Voorbeeld - v6.3 Bpve.xlsx", sheet_name="Locatiematrix", 
                            usecols="A:BB", nrows = 52)

print(df_costs.head(5))
print(df_costs.tail(5))

df_costs = df_costs.rename(columns = {'Unnamed: 0' : 'Locatie'})

print(df_costs[~df_costs['Locatie'].isin(df_ontgraven['Locatie'])]['Locatie'].unique())
print(df_costs[~df_costs['Locatie'].isin(df_aanvullen['Locatie'])]['Locatie'].unique())

def find_value(df, source, destination):
    try:
        return df.loc[df['Locatie'] == source, destination].values[0]
    except (IndexError, KeyError):
        return None

# Example usage
value = find_value(df_costs, 'COB-0055', 'Leverantie')
print(f"Value found: {value}")

        Unnamed: 0  Leverantie  Afvoer  depot  COB-0055  depot COB-0055  \
0         COB-0055           0    1.00    0.3       0.0               0   
1   depot COB-0055           0    0.00    0.0       0.0               0   
2  depot OBJ-01.01           0    0.55    0.0       0.0               0   
3  depot OBJ-01.02           0    1.50    0.0       0.0               0   
4  depot OBJ-01.03           0    1.50    0.0       0.0               0   

   depot OBJ-01.01  depot OBJ-01.02  depot OBJ-01.03  depot OBJ-01.04  ...  \
0                0                0                0                0  ...   
1                0                0                0                0  ...   
2                0                0                0                0  ...   
3                0                0                0                0  ...   
4                0                0                0                0  ...   

   OBJ-02.04  OBJ-02.05  OBJ-02.06  OBJ-02.07.01  OBJ-02.07.02  OBJ-02.08  \
0  

In [19]:

# df_costs = pd.read_excel("../Data/Demo grondstromentool.xlsx", sheet_name="Locatiematrix", 
#                              skiprows=2, usecols="F:O", nrows = 10) 

# df_costs = df_costs.set_index('Unnamed: 5')
# df_costs = df_costs.rename({'Unnamed: 5' : ''}, axis = 1)

# def get_cost(df, from_location, to_location):
#     try:
#         return df.loc[from_location, to_location]
#     except KeyError:
#         return f"One or both locations '{from_location}', '{to_location}' not found in the matrix"
    
# get_cost(df_costs, 'Afvoer per as', 'Meander de Waarden deel B')


In [20]:
print(df_aanvullen) #demand
print(df_ontgraven) #supply
print(df_compatibility) #for civil engineering applicability and priority
#
print(df_costs) #costs to move between locations
print(df_planning) #time windows for when locations are available and need to be finished

7   Object    Locatie Locatietype WB/LB     Grondsoort Erosieklasse  \
15    DS01  OBJ-01.01     Locatie    LB      teelaarde          nan   
16    DS01  OBJ-01.01     Locatie    WB      teelaarde          nan   
17    DS01  OBJ-01.01     Locatie    WB           klei            3   
18    DS01  OBJ-01.01     Locatie    WB  kernmateriaal          nan   
19    DS01  OBJ-01.01     Locatie    LB           klei            3   
..     ...        ...         ...   ...            ...          ...   
155    MDW  OBJ-03.04     Locatie    WB           zand          nan   
156    MDW  OBJ-03.04     Locatie    WB           zand           NG   
157    DUD  OBJ-03.06     Locatie    WB          grond          nan   
158    DUD  OBJ-03.06     Locatie    WB           klei            2   
159    DUD  OBJ-03.06     Locatie    WB      teelaarde          nan   

7   Milieuklasse Volume Civieltechnisch Objecttype  
15           NaN   2419       teelaarde       Dijk  
16           NaN   6042       teelaarde  

In [21]:
print(df_planning)

   Object Uitvoering_Start Uitvoering_End Grondwerk_Start Grondwerk_End
0     LDW       2025-02-01     2026-11-01      2025-04-01    2026-11-01
1     MBO       2025-03-01     2025-10-01      2025-03-01    2025-07-01
2     MDW       2025-03-01     2027-08-01      2025-04-01    2027-08-01
3     DUC       2025-05-01     2030-09-01      2025-05-01    2030-07-01
4     LOS       2025-07-01     2027-11-01      2026-02-01    2027-11-01
5     OZM       2028-06-01     2028-12-01      2026-06-01    2028-11-01
6     MBW       2027-01-01     2028-06-01      2027-01-01    2028-06-01
7     DUD       2028-06-01     2029-01-01      2028-06-01    2028-11-01
8     DAP       2028-06-01     2029-02-01      2028-06-01    2028-12-01
10   DS01       2025-11-01     2027-05-01      2026-02-01    2026-03-01
11  DS02A       2025-07-01     2027-02-01      2025-08-01    2025-11-01
12  DS02B       2025-01-01     2026-11-01      2025-01-01    2025-08-01
13   DS03       2025-09-01     2027-02-01      2025-09-01    202

In [22]:
print(df_aanvullen.head(2))
print(df_ontgraven.head(2))

7  Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
15   DS01  OBJ-01.01     Locatie    LB  teelaarde          nan          NaN   
16   DS01  OBJ-01.01     Locatie    WB  teelaarde          nan          NaN   

7  Volume Civieltechnisch Objecttype  
15   2419       teelaarde       Dijk  
16   6042       teelaarde       Dijk  
  Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
1    NaN   COB-0055     Locatie    WB       klei            3          NaN   
2   DS01  OBJ-01.01     Locatie    WB  teelaarde          nan          NaN   

     Volume Civieltechnisch Objecttype  
1 -52920.00          klei 3       Dijk  
2  -5238.64       teelaarde       Dijk  


In [23]:
print(compatibility_dict_civ)

{('Dijk', 'teelaarde_kleiinkassing'): {('Verlies', 'Transportverlies'): 1, ('Dijk', 'teelaarde_kleiinkassing'): 1, ('Rivier', 'Roofgrond'): 1, ('Rivier', 'Roofgrond flex'): 1}, ('Dijk', 'teelaarde'): {('Verlies', 'Transportverlies'): 1, ('Verlies', '10% verlies'): 1, ('Dijk', 'teelaarde'): 1, ('Rivier', 'Roofgrond'): 1, ('Rivier', 'Roofgrond flex'): 1}, ('Dijk', 'grond'): {('Verlies', 'Transportverlies'): 1, ('Dijk', 'klei 3'): 1, ('Dijk', 'kernmateriaal'): 3, ('Dijk', 'klei_kleiinkassing 3'): 2, ('Rivier', 'grond'): 4, ('Rivier', 'Roofgrond'): 1, ('Rivier', 'Roofgrond flex'): 1}, ('Dijk', 'klei 3'): {('Verlies', 'Transportverlies'): 1, ('Dijk', 'klei_kleiinkassing 3'): 1, ('Rivier', 'grond'): 1, ('Rivier', 'Roofgrond'): 1, ('Rivier', 'Roofgrond flex'): 1, ('Algemeen', 'Verkoop'): 1}, ('Dijk', 'zand NG'): {('Verlies', 'Transportverlies'): 1, ('Rivier', 'grond'): 1, ('Rivier', 'Roofgrond'): 1, ('Rivier', 'Roofgrond flex'): 1, ('Algemeen', 'Locale verschraling'): 1}, ('Rivier', 'klei 1')

In [24]:
print(df_ontgraven.tail(1))
print(df_aanvullen.head(1))

print(df_ontgraven.columns)
print(df_aanvullen.columns)

    Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
266    DUD  OBJ-03.06     Locatie    WB       klei            2   B (LB=Ind)   

      Volume Civieltechnisch Objecttype  
266 -3745.56          klei 2     Rivier  
7  Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
15   DS01  OBJ-01.01     Locatie    LB  teelaarde          nan          NaN   

7  Volume Civieltechnisch Objecttype  
15   2419       teelaarde       Dijk  
Index(['Object', 'Locatie', 'Locatietype', 'WB/LB', 'Grondsoort',
       'Erosieklasse', 'Milieuklasse', 'Volume', 'Civieltechnisch',
       'Objecttype'],
      dtype='object')
Index(['Object', 'Locatie', 'Locatietype', 'WB/LB', 'Grondsoort',
       'Erosieklasse', 'Milieuklasse', 'Volume', 'Civieltechnisch',
       'Objecttype'],
      dtype='object', name=7)


In [25]:
print(compatibility_dict_env)

{('Dijk', 'LB', 'Industrie (WB = A)'): [('Dijk', 'LB', 'Industrie'), ('Dijk', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'B (LB = Ind)'), ('Rivier', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'AW (LB = Ind)'), ('Rivier', 'WB', 'AW')], ('Dijk', 'WB', 'A (LB = Ind)'): [('Dijk', 'LB', 'Industrie'), ('Dijk', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'B (LB = Ind)'), ('Rivier', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'AW (LB = Ind)'), ('Rivier', 'WB', 'AW'), ('Algemeen', None, 'Verkoop')], ('Rivier', 'WB', 'NT'): [('Algemeen', None, 'Stort')], ('Rivier', 'WB', 'B (LB = Ind)'): [('Dijk', 'LB', 'Industrie'), ('Dijk', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'B (LB = Ind)'), ('Rivier', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'AW (LB = Ind)'), ('Rivier', 'WB', 'AW'), ('Algemeen', None, 'Zandbanen'), ('Algemeen', None, 'Locale verschraling')], ('Rivier', 'WB', 'A (LB = Ind)'): [('Dijk', 'LB', 'Industrie'), ('Dijk', 'WB', 'A (LB = Ind)'), ('Rivier', 'WB', 'B (LB = Ind)'), ('Rivier', 'WB', 'A (LB = Ind)'),

In [26]:
df_ontgraven[df_ontgraven['Civieltechnisch'] == 'Keramische klei']

,Object,Locatie,Locatietype,WB/LB,Grondsoort,Erosieklasse,Milieuklasse,Volume,Civieltechnisch,Objecttype
214,DUC,OBJ-03.02,Locatie,WB,klei,Keramische klei,NaN,-775000.0,Keramische klei,Rivier


In [27]:
print(len(df_ontgraven))

df_ontgraven = df_ontgraven[df_ontgraven['Civieltechnisch'] != 'Keramische klei']

print(len(df_ontgraven))

266
265


In [28]:
df_ontgraven[df_ontgraven['Locatie'] == 'COB-0055']

,Object,Locatie,Locatietype,WB/LB,Grondsoort,Erosieklasse,Milieuklasse,Volume,Civieltechnisch,Objecttype
1,NaN,COB-0055,Locatie,WB,klei,3,NaN,-52920.0,klei 3,Dijk


In [ ]:
# Create a dictionary with all the objects
data_objects = {
    'df_aanvullen': df_aanvullen,
    'df_ontgraven': df_ontgraven,
    #'df_compatibility': df_compatibility,
    'compatibility_dict_env': compatibility_dict_env,
    'compatibility_dict_civ': compatibility_dict_civ,
    'df_costs': df_costs,
    'df_planning': df_planning
}

# Save all objects to a single file
with open('..\Data\soil_transport_data.pkl', 'wb') as file:
    pickle.dump(data_objects, file)

In [30]:
print(df_ontgraven.columns)
print(df_aanvullen.columns)

Index(['Object', 'Locatie', 'Locatietype', 'WB/LB', 'Grondsoort',
       'Erosieklasse', 'Milieuklasse', 'Volume', 'Civieltechnisch',
       'Objecttype'],
      dtype='object')
Index(['Object', 'Locatie', 'Locatietype', 'WB/LB', 'Grondsoort',
       'Erosieklasse', 'Milieuklasse', 'Volume', 'Civieltechnisch',
       'Objecttype'],
      dtype='object', name=7)


In [31]:
print(df_ontgraven[df_ontgraven['Civieltechnisch'] == 'klei'])

    Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
125    LDW  OBJ-02.05     Locatie    WB       klei          nan           NT   
146    LOS  OBJ-02.06     Locatie    WB       klei          nan           NT   
250    MDW  OBJ-03.04     Locatie    WB       klei          nan           NT   

       Volume Civieltechnisch Objecttype  
125  -9331.56            klei     Rivier  
146 -63834.26            klei     Rivier  
250 -26152.28            klei     Rivier  


In [32]:
union_template = pd.concat([df_ontgraven, df_aanvullen], keys=['Ontgraven', 'Aanvullen'], names=['Ontgraven/Aanvullen', 'Index'])

union_template = union_template.reset_index(level=0)  # Reset the index for the 'Type' level

union_template = union_template.reset_index(drop=True)  # Reset the index to avoid multi-level index

print(union_template.head(266))

union_template.to_csv('../Data/union_template.csv')

    Ontgraven/Aanvullen Object    Locatie Locatietype WB/LB Grondsoort  \
0             Ontgraven    NaN   COB-0055     Locatie    WB       klei   
1             Ontgraven   DS01  OBJ-01.01     Locatie    WB  teelaarde   
2             Ontgraven   DS01  OBJ-01.01     Locatie    LB  teelaarde   
3             Ontgraven   DS01  OBJ-01.01     Locatie    WB      grond   
4             Ontgraven   DS01  OBJ-01.01     Locatie    LB      grond   
..                  ...    ...        ...         ...   ...        ...   
261           Ontgraven    DUD  OBJ-03.06     Locatie    WB       klei   
262           Ontgraven    DUD  OBJ-03.06     Locatie    WB       klei   
263           Ontgraven    DUD  OBJ-03.06     Locatie    WB       klei   
264           Ontgraven    DUD  OBJ-03.06     Locatie    WB       klei   
265           Aanvullen   DS01  OBJ-01.01     Locatie    LB  teelaarde   

    Erosieklasse Milieuklasse   Volume Civieltechnisch Objecttype  
0              3          NaN -52920.0     

In [33]:
grouping_cols = union_template.columns.to_list()

grouping_cols.remove('Volume')

# Convert all grouping columns to string type
for col in grouping_cols:
    union_template[col] = union_template[col].astype(str).str.strip()

# Now try the groupby operation again
print(union_template.groupby(grouping_cols).size().sort_values(ascending=False))
print(len(union_template))

# union_template = union_template.groupby(grouping_cols).agg({'Volume': 'sum'}).reset_index()

# print(len(union_template))

Ontgraven/Aanvullen  Object  Locatie    Locatietype  WB/LB  Grondsoort          Erosieklasse  Milieuklasse  Civieltechnisch       Objecttype
Aanvullen            MDW     OBJ-03.04  Locatie      WB     zand                nan           nan           zand NG               Rivier        2
Ontgraven            MDW     OBJ-03.04  Locatie      WB     klei                NG            B (LB=Ind)    klei NG               Rivier        2
Aanvullen            DS02B   OBJ-01.03  Locatie      LB     kernmateriaal       3             nan           klei 3                Dijk          1
                                                            klei                3             nan           klei 3                Dijk          1
                                                            teelaarde           nan           Pipingberm    teelaarde             Dijk          1
                                                                                                                                 

In [34]:
df_aanvullen[(df_aanvullen['Civieltechnisch'] == 'klei 1') & (df_aanvullen['Objecttype'] == 'Dijk')]

7,Object,Locatie,Locatietype,WB/LB,Grondsoort,Erosieklasse,Milieuklasse,Volume,Civieltechnisch,Objecttype


In [44]:
print(df_ontgraven[2:10])

   Object    Locatie Locatietype WB/LB Grondsoort Erosieklasse Milieuklasse  \
3    DS01  OBJ-01.01     Locatie    LB  teelaarde          nan          NaN   
4    DS01  OBJ-01.01     Locatie    WB      grond          nan          NaN   
5    DS01  OBJ-01.01     Locatie    LB      grond          nan          NaN   
6   DS02A  OBJ-01.02     Locatie    WB  teelaarde          nan          NaN   
7   DS02A  OBJ-01.02     Locatie    LB  teelaarde          nan          NaN   
8   DS02A  OBJ-01.02     Locatie    WB      grond          nan          NaN   
9   DS02B  OBJ-01.03     Locatie    WB  teelaarde          nan          NaN   
10  DS02B  OBJ-01.03     Locatie    LB  teelaarde          nan          NaN   

      Volume Civieltechnisch Objecttype  
3   -1906.96       teelaarde       Dijk  
4   -9096.36           grond       Dijk  
5   -3914.12           grond       Dijk  
6   -8650.40       teelaarde       Dijk  
7    -415.36       teelaarde       Dijk  
8   -6468.98           grond       D